# ALGOPACK-only baseline

Полностью отдельный pipeline на премиум-данных MOEX ALGOPACK +
открытые ISS-календари + дивиденды. Никаких regular-OHLCV, никаких
SMA/RSI/MACD, никакого Brent/CBR/ticker-dummies — только то, что
даёт **подписка**.

## Источники

* **TradeStats** (`pr_open/high/low/close/std/vol/val/trades/pr_vwap`) —
  заменяет regular candles. Это и есть «новые свечи»: те же OHLCV,
  но built **на стороне MOEX** из их audit-логов сделок (точнее
  чем resampling 1-мин).
* **TradeStats split** (`vol_b/s, val_b/s, trades_b/s, pr_vwap_b/s, disb`) —
  aggressive buy/sell разбивка → главный microstructure-сигнал.
* **OrderStats** — order flow imbalance (заявки выставленные/снятые).
* **OBStats** — спрэды и order book imbalance (микропrice сигнал).
* **Calendars** — флаги торговых/нерабочих дней, дни до экспирации,
  суспендирование.
* **Dividends** — days_to_ex_date, ex-day flag, value_last/next/yield.

## Структура

1. Окружение, Drive, проверка наличия данных в `Algopack/`.
2. Load всех источников + sanity-check на размеры.
3. Build feature frame: TradeStats как backbone + merge остальных.
4. Train (тот же TimeXer + auto pos_weight).
5. Inference + Bayes/Conformal + threshold sweep.


## 0. Окружение

In [ ]:
from __future__ import annotations

import os
import subprocess
from pathlib import Path

try:
    import google.colab  # type: ignore  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_OWNER = 'Pozdnyakof'
REPO_NAME = 'lstm_exchange_predictor'
REPO_BRANCH = 'main'

if IN_COLAB:
    PROJECT_ROOT = Path('/content') / REPO_NAME
    env = os.environ.copy()
    env['GIT_TERMINAL_PROMPT'] = '0'
    if PROJECT_ROOT.exists():
        subprocess.run(['git', '-C', str(PROJECT_ROOT), 'fetch', 'origin', REPO_BRANCH],
                       check=True, env=env)
        subprocess.run(['git', '-C', str(PROJECT_ROOT), 'reset', '--hard',
                        f'origin/{REPO_BRANCH}'], check=True, env=env)
        head = subprocess.run(
            ['git', '-C', str(PROJECT_ROOT), 'rev-parse', '--short', 'HEAD'],
            capture_output=True, text=True, check=True,
        ).stdout.strip()
        print(f'HEAD={head}')
    else:
        subprocess.run(
            ['git', 'clone', '--depth', '1', '-b', REPO_BRANCH,
             f'https://github.com/{REPO_OWNER}/{REPO_NAME}.git', str(PROJECT_ROOT)],
            check=True, env=env,
        )
else:
    PROJECT_ROOT = Path.cwd().parent

PROJECT_ROOT = PROJECT_ROOT.resolve()
print(f'PROJECT_ROOT: {PROJECT_ROOT}, IN_COLAB: {IN_COLAB}')


In [ ]:
if IN_COLAB:
    subprocess.run(
        ['pip', 'install', '--quiet',
         'torch>=2.2', 'pandas>=2.1', 'pyarrow>=15.0',
         'scikit-learn>=1.4', 'requests>=2.31', 'tqdm>=4.66', 'ipywidgets>=8.0'],
        check=True,
    )
    print('Готово.')


In [ ]:
import sys

SRC = PROJECT_ROOT / 'src'
if not (SRC / 'graduate_work' / '__init__.py').exists():
    raise FileNotFoundError(f'graduate_work не найден в {SRC}')
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s: %(message)s')

import dataclasses

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from graduate_work.config import default_config

cfg = default_config()
print('Тикеры:', cfg.data.tickers)
print('Горизонты:', cfg.data.horizons)
print('Окно:', cfg.data.window_size)


## 1. Drive: данные ALGOPACK + календари + дивиденды

Ожидаемая структура на Drive:

```
MyDrive/lstm_exchange/
└── data/raw/Algopack/
    ├── algopack/
    │   ├── tradestats/<ticker>.csv
    │   ├── orderstats/<ticker>.csv
    │   └── obstats/<ticker>.csv
    ├── calendars/<product>.csv  (12 файлов)
    └── dividends/<ticker>.csv
```


In [ ]:
import shutil

DRIVE_BASE = Path('/content/drive/MyDrive/lstm_exchange')

if IN_COLAB:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive', force_remount=False)
    src_dir = DRIVE_BASE / 'data' / 'raw' / 'Algopack'
    if not src_dir.exists():
        # Backwards compat: возможно лежит в обычной data/raw/.
        src_dir = DRIVE_BASE / 'data' / 'raw'
    if src_dir.exists():
        shutil.copytree(src_dir, cfg.paths.data_raw, dirs_exist_ok=True)
        print(f'Подтянули данные из {src_dir}')
    else:
        raise FileNotFoundError(
            f'Не найдено {src_dir}. Сначала запусти '
            'scripts/08_download_algopack.py локально и положи папку '
            'data/raw/ как data/raw/Algopack/ на Drive.'
        )

# Проверяем что есть.
for sub in ['algopack/tradestats', 'algopack/orderstats', 'algopack/obstats',
            'calendars', 'dividends']:
    p = cfg.paths.data_raw / sub
    if p.exists():
        n = sum(1 for _ in p.glob('*.csv'))
        print(f'  {sub}: {n} файлов')
    else:
        print(f'  {sub}: ОТСУТСТВУЕТ')


## 2. Загрузка SuperCandles + календарей + дивидендов

In [ ]:
def _load_csv(path: Path) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame()
    df = pd.read_csv(path)
    return df


def _load_supercandle(product: str, ticker: str) -> pd.DataFrame:
    p = cfg.paths.data_raw / 'algopack' / product / f'{ticker}.csv'
    df = _load_csv(p)
    if df.empty or 'tradedate' not in df.columns:
        return df
    if 'tradetime' in df.columns:
        ts = pd.to_datetime(
            df['tradedate'].astype(str) + ' ' + df['tradetime'].astype(str),
            utc=True, errors='coerce',
        )
    else:
        ts = pd.to_datetime(df['tradedate'], utc=True, errors='coerce')
    df.index = ts
    df.index.name = 'begin'
    return df.dropna(how='all').sort_index()


# Грузим всё в словари: {ticker: DataFrame}.
ts_data, os_data, ob_data = {}, {}, {}
div_data = {}
for ticker in cfg.data.tickers:
    ts_data[ticker] = _load_supercandle('tradestats', ticker)
    os_data[ticker] = _load_supercandle('orderstats', ticker)
    ob_data[ticker] = _load_supercandle('obstats', ticker)
    p = cfg.paths.data_raw / 'dividends' / f'{ticker}.csv'
    if p.exists():
        df = pd.read_csv(p)
        if 'registryclosedate' in df.columns:
            df['registryclosedate'] = pd.to_datetime(
                df['registryclosedate'], utc=True, errors='coerce',
            )
        div_data[ticker] = df

# Календари — все 12 продуктов.
cal_dir = cfg.paths.data_raw / 'calendars'
calendars = {p.stem: pd.read_csv(p) for p in cal_dir.glob('*.csv')} if cal_dir.exists() else {}

print('TradeStats:')
for t, df in ts_data.items():
    print(f'  {t}: {len(df)} 5-мин баров, range {df.index.min()} .. {df.index.max() if not df.empty else "-"}')
print(f'Calendars: {list(calendars.keys())}')
print(f'Dividends: {list(div_data.keys())}')


In [ ]:
# --- Schema diagnostic: видим РЕАЛЬНЫЕ колонки в каждом источнике ---
# Полезно сравнить с документированной схемой ALGOPACK.
from graduate_work.features.algopack_features import print_schema

if cfg.data.tickers:
    sample = cfg.data.tickers[0]
    print(f'=== SCHEMA для тикера {sample} ===')
    print_schema(
        tradestats=ts_data.get(sample),
        orderstats=os_data.get(sample),
        obstats=ob_data.get(sample),
    )
    print()
    # Проверяем плотность данных: 105 баров/день в session 07:00-15:45 UTC.
    if not ts_data[sample].empty:
        ts = ts_data[sample]
        n_dates = ts.index.normalize().nunique()
        avg_per_day = len(ts) / max(n_dates, 1)
        print(f'TradeStats {sample}: {len(ts)} баров, {n_dates} уникальных дат, '
              f'avg {avg_per_day:.1f} баров/день')
        print(f'  (ожидание ~105 баров/день — сессия 07:00-15:45 UTC)')
        if avg_per_day < 50:
            print('WARNING: данные разреженные. Проверь pagination или date filter.')


## 3. Build feature frame

* **TradeStats** даёт OHLCV (`pr_*`, `vol`, `val`, `trades`, `pr_vwap`) +
  agression split.
* **OrderStats** + **OBStats** мержатся per-bar по timestamp.
* **Calendars** + **Dividends** добавляются как event-фичи.


In [ ]:
from graduate_work.features.algopack_features import (
    obstats_features, orderstats_features, tradestats_features, order_to_trade_ratio,
)
from graduate_work.features.calendar_features import (
    dividend_features, trading_day_features, expirations_features,
)
from graduate_work.features.targets import cost_aware_classification_labels


def _ts_to_ohlcv(ts: pd.DataFrame, ticker: str) -> pd.DataFrame:
    """TradeStats pr_* → стандартный OHLCV-like frame с колонками open/high/low/close/volume/value/ticker."""
    if ts.empty:
        return pd.DataFrame()
    out = pd.DataFrame(index=ts.index)
    out['open'] = ts['pr_open'].astype(float)
    out['high'] = ts['pr_high'].astype(float)
    out['low']  = ts['pr_low'].astype(float)
    out['close']= ts['pr_close'].astype(float)
    out['volume'] = ts['vol'].astype(float)
    out['value']  = ts['val'].astype(float)
    out['vwap']   = ts['pr_vwap'].astype(float)
    out['ticker'] = ticker
    return out


def _build_log_returns(close: pd.Series) -> pd.DataFrame:
    """Минимальный набор лог-доходностей разных лагов — единственная техническая фича."""
    lr = np.log(close / close.shift(1))
    return pd.DataFrame({
        'lr_1':  lr,
        'lr_5':  lr.rolling(5).sum(),
        'lr_15': lr.rolling(15).sum(),
        'lr_30': lr.rolling(30).sum(),
    }, index=close.index)


def build_ticker_frame(ticker: str) -> pd.DataFrame:
    ts_df = ts_data[ticker]
    if ts_df.empty:
        return pd.DataFrame()
    ohlcv = _ts_to_ohlcv(ts_df, ticker)
    feat = ohlcv.copy()
    feat = feat.join(_build_log_returns(feat['close']), how='left')
    # Microstructure из TradeStats.
    feat = feat.join(tradestats_features(ts_df), how='left')
    # OrderStats / OBStats — outer-merge по индексу.
    if not os_data[ticker].empty:
        feat = feat.join(orderstats_features(os_data[ticker]), how='left')
    if not ob_data[ticker].empty:
        feat = feat.join(obstats_features(ob_data[ticker]), how='left')
    if not os_data[ticker].empty and not ts_data[ticker].empty:
        feat = feat.join(order_to_trade_ratio(os_data[ticker], ts_data[ticker]), how='left')
    # Календарные фичи.
    trading_days = calendars.get('trading_days_stock', pd.DataFrame())
    feat = feat.join(trading_day_features(trading_days, feat.index), how='left')
    expir = calendars.get('futures_expirations', pd.DataFrame())
    feat = feat.join(expirations_features(expir, feat.index), how='left')
    # Дивиденды (если есть).
    if ticker in div_data and not div_data[ticker].empty:
        last_close = float(feat['close'].iloc[-1]) if not feat['close'].empty else None
        feat = feat.join(
            dividend_features(div_data[ticker], feat.index, last_close_price=last_close),
            how='left',
        )
    feat = feat.fillna(0.0)
    return feat


frames = []
for ticker in cfg.data.tickers:
    f = build_ticker_frame(ticker)
    if not f.empty:
        frames.append(f)
        print(f'  {ticker}: {f.shape}')

full = pd.concat(frames).sort_index()
feature_cols = [c for c in full.columns if c not in ('ticker',)]
print(f'\nfull frame: {full.shape}, feature_cols={len(feature_cols)}')
print(f'feature_cols sample: {feature_cols[:10]}...')


## 4. Targets, splits, training

Применяем ту же cost-aware classification, тот же TimeXer + auto pos_weight,
но **на ALGOPACK feature_frame**. Если результат окажется лучше regular
baseline — это сильный аргумент в защиту: премиум-данные дают edge,
которого нет в открытых OHLCV.


In [ ]:
def _attach_targets(frame: pd.DataFrame) -> tuple[pd.DataFrame, list[str]]:
    """Добавить cost-aware classification labels по close-цене per-ticker."""
    out_parts = []
    for ticker, sub in frame.groupby('ticker'):
        labels = cost_aware_classification_labels(
            open_price=sub['open'],
            close_price=sub['close'],
            horizons=cfg.data.horizons,
            entry_cost=cfg.trading.commission_rate + cfg.trading.slippage_rate,
            exit_cost=cfg.trading.commission_rate + cfg.trading.slippage_rate,
            label_smoothing=cfg.data.label_smoothing,
            direction='long',
        )
        merged = sub.join(labels, how='left')
        out_parts.append(merged)
    out = pd.concat(out_parts).sort_index()
    target_cols = [f'target_h{h}' for h in cfg.data.horizons]
    return out, target_cols


full_with_targets, target_cols = _attach_targets(full)
print(f'frame с таргетами: {full_with_targets.shape}')
print(f'target_cols: {target_cols}')
for c in target_cols:
    p_up = float((full_with_targets[c].dropna() >= 0.5).mean())
    print(f'  {c}: P(UP)={p_up:.3f}')

# feature_cols обновляем — теперь включают targets, поэтому пере-определяем
# scaler-input как feature_cols MINUS target_cols.
feature_cols = [c for c in full_with_targets.columns
                if c not in ('ticker',) and c not in target_cols and not c.startswith('lr_h')]
print(f'
feature_cols обновлено: {len(feature_cols)} (без targets и raw lr_h*)')


In [ ]:
# Hronological split + scaling + windowing.
from graduate_work.features.pipeline import chronological_split, _build_arrays
from graduate_work.features.scaler import StandardScaler

scaler_feature_cols = [c for c in feature_cols if c not in target_cols]
train_df, val_df, test_df = chronological_split(
    full_with_targets,
    cfg.data.train_ratio,
    cfg.data.val_ratio,
    purge_horizon=max(cfg.data.horizons),
)

scaler = StandardScaler()
scaler.fit(train_df, scaler_feature_cols)
train_df = scaler.transform(train_df)
val_df = scaler.transform(val_df)
test_df = scaler.transform(test_df)

train = _build_arrays(train_df, scaler_feature_cols, target_cols, cfg.data.window_size, desc='train')
val   = _build_arrays(val_df,   scaler_feature_cols, target_cols, cfg.data.window_size, desc='val')
test  = _build_arrays(test_df,  scaler_feature_cols, target_cols, cfg.data.window_size, desc='test')
print(f'train: {train["x"].shape}, val: {val["x"].shape}, test: {test["x"].shape}')


In [ ]:
from graduate_work.model import build_model
from graduate_work.training import Trainer, mc_predict, set_seed

set_seed(cfg.training.seed)
model = build_model(
    input_dim=train['x'].shape[-1],
    num_horizons=len(cfg.data.horizons),
    cfg=cfg.model,
)
trainer = Trainer(
    model, cfg.training,
    data_cfg=cfg.data, trading_cfg=cfg.trading,
)
history = trainer.fit(train, val)
print(f'Best epoch: {int(np.argmin(history.val_loss))+1} / {len(history.train_loss)}, '
      f'best val_loss={float(min(history.val_loss)):.5f}')


## 5. Inference + Bayes/Conformal + threshold sweep

In [ ]:
from graduate_work.backtest import compute_metrics, run_backtest
from graduate_work.backtest.engine import prices_from_full_frame
from graduate_work.strategy import (
    ConformalSignalGenerator, SignalGenerator,
    attach_actual_targets, attach_lr_targets,
    build_predictions_frame, calibrate_bayes_threshold,
)

is_cls = cfg.data.mode == 'classification'
mean, std = mc_predict(
    model, test['x'],
    mc_passes=cfg.training.mc_passes, batch_size=cfg.training.batch_size,
    apply_sigmoid=is_cls,
)
val_mean, val_std = mc_predict(
    model, val['x'],
    mc_passes=cfg.training.mc_passes, batch_size=cfg.training.batch_size,
    apply_sigmoid=is_cls,
)
print(f'Test: prob mean={mean.mean():.4f} max={mean.max():.4f} std mean={std.mean():.4f}')

predictions = build_predictions_frame(
    test['timestamp'], test['ticker'], mean, std, cfg.data.horizons,
)
val_predictions = build_predictions_frame(
    val['timestamp'], val['ticker'], val_mean, val_std, cfg.data.horizons,
)


In [ ]:
# Готовим test_prices из ALGOPACK-данных (open/close из TradeStats).
test_prices = full_with_targets.loc[
    (full_with_targets.index >= pd.to_datetime(min(test['timestamp']), utc=True))
].copy()
# prices_from_full_frame ожидает open + close + ticker.
test_prices = test_prices[['open', 'close', 'ticker']]
bars_per_year = cfg.data.bars_per_year

# Bayes calibration.
cost_per_trade = 2.0 * (cfg.trading.commission_rate + cfg.trading.slippage_rate)
lr_targets = attach_lr_targets(full_with_targets, val, cfg.data.horizons)
bayes_calib = calibrate_bayes_threshold(val_predictions, lr_targets, cost_per_trade=cost_per_trade)
tc_bayes = dataclasses.replace(
    cfg.trading,
    probability_threshold=bayes_calib.min_expected_return,
    selection_correction='none',
)
bayes_signals = SignalGenerator(tc_bayes, mode=cfg.data.mode).generate(predictions)
bayes_bt = run_backtest(bayes_signals, test_prices, tc_bayes)
bayes_m = compute_metrics(bayes_bt.equity, bayes_bt.trades, periods_per_year=bars_per_year)
print(f'Bayes T={bayes_calib.min_expected_return:.4f}  '
      f'BUY={int((bayes_signals["action"]=="BUY").sum())}  '
      f'sharpe={bayes_m["sharpe"]:.3f}  '
      f'return={bayes_m["total_return"]*100:.2f}%')

# Conformal directional.
conf_gen = ConformalSignalGenerator(cfg.trading, alpha=0.1, directional=True)
val_targets = attach_actual_targets(val, cfg.data.horizons)
conf_calib = conf_gen.calibrate(val_predictions, val_targets)
conf_signals = conf_gen.generate(predictions)
conf_bt = run_backtest(conf_signals, test_prices, cfg.trading)
conf_m = compute_metrics(conf_bt.equity, conf_bt.trades, periods_per_year=bars_per_year)
print(f'Conformal T={conf_calib.threshold:.4f}  '
      f'BUY={int((conf_signals["action"]=="BUY").sum())}  '
      f'sharpe={conf_m["sharpe"]:.3f}  '
      f'return={conf_m["total_return"]*100:.2f}%')


In [ ]:
# --- Threshold sweep (Bayes-style, Šidák=none) ---
thresholds = [0.51, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85]
sweep_rows = []
for thr in thresholds:
    tc = dataclasses.replace(
        cfg.trading,
        probability_threshold=float(thr),
        selection_correction='none',
    )
    sigs = SignalGenerator(tc, mode=cfg.data.mode).generate(predictions)
    n_buy = int((sigs['action'] == 'BUY').sum())
    if n_buy == 0:
        sweep_rows.append({'T': thr, 'n_buy': 0, 'n_trades': 0,
                           'sharpe': 0.0, 'total_return': 0.0,
                           'final_equity': float(cfg.trading.initial_capital),
                           'win_rate': 0.0, 'max_dd': 0.0})
        continue
    bt = run_backtest(sigs, test_prices, tc)
    m = compute_metrics(bt.equity, bt.trades, periods_per_year=bars_per_year)
    sweep_rows.append({
        'T': thr, 'n_buy': n_buy, 'n_trades': int(m['n_trades']),
        'sharpe': float(m['sharpe']),
        'total_return': float(m['total_return']),
        'final_equity': float(m['final_equity']),
        'win_rate': float(m['win_rate']),
        'max_dd': float(m['max_drawdown']),
    })

sweep = pd.DataFrame(sweep_rows)
print('=== ALGOPACK BASELINE — THRESHOLD SWEEP ===')
print(sweep.to_string(index=False, float_format=lambda x: f'{x:.4f}'))


## Сравнение с regular baseline

Чтобы оценить вклад ALGOPACK-данных, сравни итоговые `sharpe` и
`total_return` с теми же метриками из `training_pipeline.ipynb`
(который работает на regular MOEX OHLCV + индикаторах).

Если **`prob_max` выше**, **`bayes_n_buy` ненулевой** и **sharpe
положительный** при той же модели — это прямое доказательство, что
**микроструктурные данные ALGOPACK решают проблему prediction
collapse**, которой страдал regular baseline. Это самая сильная
теза для ВКР.
